# SEMMA Project: Early-Warning Prediction of Student Final Grades

**Methodology:** SAS **SEMMA** — **S**ample · **E**xplore · **M**odify · **M**odel · **A**ssess
**Dataset:** UCI *Student Performance* (Cortez & Silva, 2008), Kaggle mirror `uciml/student-alcohol-consumption` — two Portuguese secondary schools; **395 math students** (primary), 649 Portuguese-language students (used for a transfer check). Target: `G3` = final grade (0–20).
**Course:** SJSU Data Mining — assignment on CRISP-DM · SEMMA · KDD (this is the SEMMA project; one dataset per methodology).

Run in Google Colab: *Runtime → Run all*. The notebook downloads the data from UCI if `data/` is missing.

> ### Honest scope statement
> **I did not run SAS.** I had no SAS Enterprise Miner / SAS Viya trial licence. This notebook is a **Python re-implementation of the SAS SEMMA workflow**, node by node (mapping table below), using scikit-learn. Numbers here are *not* SAS output, and the SAS-style score code written in Phase A has **not** been executed in SAS (its arithmetic is verified in Python only).

## The design question that drives this project: the G1/G2 problem
`G1` and `G2` are the first- and second-period grades. `G3` is strongly predicted by them (they are the same student's earlier marks), so a model that uses them looks superb but is *nearly useless*: if you already know G2, you are almost done. The decision-relevant question is **"which students will fail, *before* we have grades?"** Therefore every stage of the pipeline is run on **three information scenarios**, and results are never reported for only one:

| Scenario | Inputs | Use case |
|---|---|---|
| **A — early warning** | demographics, family, study habits, absences (no grades) | start of year — the meaningful case |
| **B — after period 1** | A + `G1` | mid-year intervention |
| **C — after period 2** | A + `G1` + `G2` | "leaky-looking" upper bound; sanity check |

Baselines per scenario: *predict the training mean*; and *copy the latest known grade* (B: `G1`, C: `G2`).

## SEMMA ⇄ SAS Enterprise Miner / Viya node mapping
| SEMMA stage | Task here | SAS EM node (or Viya Model Studio equivalent) | Python used |
|---|---|---|---|
| Sample | Load, assign variable roles/levels | **Input Data Source** | pandas, role table |
| Sample | 60/20/20 stratified partition, representativeness | **Data Partition** (stratified on target) | `train_test_split(stratify=)` |
| Explore | Distributions, missing, outliers | **StatExplore**, **MultiPlot** | pandas, seaborn |
| Explore | Variable worth (chi-square, MI, Kruskal) | **StatExplore** (variable worth), **Variable Selection** | scipy, sklearn |
| Modify | Impute / transform / bin / derive | **Impute**, **Transform Variables**, **Interactive Binning** | sklearn pipelines |
| Modify | Variable clustering, selection | **Variable Clustering**, **Variable Selection** (LARS/Lasso) | scipy hierarchy, LassoCV |
| Model | Linear / ridge / lasso regression | **Regression**, **LARS** | sklearn |
| Model | Decision tree, forest, boosting | **Decision Tree**, **HP Forest**, **Gradient Boosting** | sklearn |
| Assess | Compare on validation (ASE, RMSE, lift) | **Model Comparison** | custom tables |
| Assess | Score new data | **Score** node (score code) | joblib + SAS-style score-code export |

## How to repeat this in SAS EM / SAS Viya (not done here)
1. Create library → register `student-mat.csv` (delimiter `;` if from UCI). Input Data Source: set `G3` role *Target* (interval); for the pass/fail variant derive `fail = (G3<10)` in a SAS Code node and set it as binary target.
2. **Data Partition** 60/20/20, *stratified* on the target; seed 42.
3. **StatExplore** + **MultiPlot** on the training role; keep a *Metadata* node to reject `G1`,`G2` for scenario A (repeat the diagram with them as inputs for B/C).
4. **Impute** (none needed, mirrors the pipeline), **Transform Variables** (`log(1+absences)`), **Variable Clustering**.
5. **Regression** (stepwise/lasso selection, validation error), **Decision Tree**, **Gradient Boosting**, **HP Forest**; connect all to **Model Comparison** with *Selection Statistic = Valid: Average Squared Error* (the champion rule I used).
6. **Score** node → export score code; compare to `artifacts/score_code_*.sas.txt` here.

## Critic loop
Critic rounds (LLM prompted as a senior SEMMA authority, per assignment) are logged in `critic_log/`; this section will be filled in from those real reviews — nothing has been fabricated here.

In [ ]:
import os, sys, json, warnings, zipfile, io, urllib.request, datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)

for d in ["data", "artifacts", "src", "deploy"]: os.makedirs(d, exist_ok=True)
MAT, POR = "data/student-mat.csv", "data/student-por.csv"
UCI_ZIP = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

def read_student(path):
    d = pd.read_csv(path)                       # Kaggle mirror is comma-separated
    if d.shape[1] == 1: d = pd.read_csv(path, sep=";")   # UCI original is semicolon-separated
    return d

if not (os.path.exists(MAT) and os.path.exists(POR)):
    try:
        print("Downloading from UCI ...")
        outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(UCI_ZIP, timeout=60).read()))
        inner = zipfile.ZipFile(io.BytesIO(outer.read("student.zip")))   # UCI zip contains a nested student.zip
        for n in ["student-mat.csv", "student-por.csv"]:
            open("data/" + n, "wb").write(inner.read(n))
    except Exception as e:
        print("Download failed (", e, ") -> put student-mat.csv / student-por.csv (Kaggle: uciml/student-alcohol-consumption) in ./data and re-run.")
print("Python", sys.version.split()[0], "| pandas", pd.__version__)

---
# S — SAMPLE
Goal of the stage: obtain a *representative*, *documented* working sample and fix a **partition** so that every later decision (exploration, selection, tuning) can be audited against data the model has not seen.

## S.1 Input Data Source — read the data and assign roles and levels

In [ ]:
mat = read_student(MAT); por = read_student(POR)
print("math:", mat.shape, "| portuguese:", por.shape)
mat.head()

In [ ]:
levels = {}
for c in mat.columns:
    if c in ["G1","G2","G3"]: levels[c] = "interval (0-20)"
    elif mat[c].dtype == object or str(mat[c].dtype) in ("str", "string"): levels[c] = "binary" if mat[c].nunique() == 2 else "nominal"
    elif mat[c].nunique() <= 5 and c != "failures": levels[c] = "ordinal (1-5)"
    else: levels[c] = "interval/count"
roles = pd.DataFrame({"role": ["TARGET" if c == "G3" else ("INPUT (conditional: scenario B/C)" if c in ["G1","G2"] else "INPUT") for c in mat.columns],
                      "level": pd.Series(levels), "n_unique": mat.nunique(), "n_missing": mat.isna().sum(), "example": mat.iloc[0]})
display(roles)

**Notes on the roles.** There is **no ID variable** — the UCI file has no student identifier, so I cannot check for duplicated students *within* a file. There *is* overlap **between** the two files: the dataset's author provides a merge key (13 attributes) for students who attended both courses. That overlap matters for the cross-subject check in Phase A.

In [ ]:
KEY = ["school","sex","age","address","famsize","Pstatus","Medu","Fedu","Mjob","Fjob","reason","nursery","internet"]
both = mat.merge(por, on=KEY, suffixes=("_mat","_por"))
print("rows matched by the author's merge key:", len(both), "(author reports 382; key is not guaranteed unique)")
print("exact duplicate rows inside math:", mat.duplicated().sum(), "| inside portuguese:", por.duplicated().sum())

## S.2 Target definition and balance
Two targets from the same variable: **regression** on `G3` and **classification** `fail = 1 if G3 < 10` (Portuguese pass mark is 10/20; `fail` is the *positive/at-risk* class).

In [ ]:
PASS_MARK = 10
mat["fail"] = (mat["G3"] < PASS_MARK).astype(int)
print("fail rate (math): %.3f   | mean G3 %.2f  sd %.2f" % (mat.fail.mean(), mat.G3.mean(), mat.G3.std()))
print("G3 == 0 (students with no final mark, i.e. dropped out / did not sit):", (mat.G3 == 0).sum(),
      "| of those, G2 > 0:", ((mat.G3 == 0) & (mat.G2 > 0)).sum())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
sns.histplot(mat.G3, bins=21, ax=ax[0]); ax[0].axvline(PASS_MARK - .5, color="red", ls="--"); ax[0].set_title("G3 (red = pass mark)")
mat.fail.value_counts().sort_index().plot.bar(ax=ax[1], color=["#4C78A8", "#E45756"]); ax[1].set_title("fail (1 = at risk)")
plt.tight_layout(); plt.savefig("artifacts/target_distribution.png", dpi=120); plt.show()

## S.3 Data Partition — 60 / 20 / 20, stratified on `fail`
SAS EM's Data Partition node splits into TRAIN / VALIDATE / TEST. With n = 395 a *random* partition can by luck put noticeably more failures in one part, so I stratify on `fail`. The next cell quantifies how much that matters instead of assuming it.

In [ ]:
from sklearn.model_selection import train_test_split
def partition(df, strat=True, seed=SEED):
    s = df["fail"] if strat else None
    tr, rest = train_test_split(df.index, test_size=0.4, stratify=s, random_state=seed)
    va, te = train_test_split(rest, test_size=0.5, stratify=(df.loc[rest, "fail"] if strat else None), random_state=seed)
    return np.array(tr), np.array(va), np.array(te)

idx_tr, idx_va, idx_te = partition(mat, strat=True)
print("sizes  train/valid/test:", len(idx_tr), len(idx_va), len(idx_te))
print("fail rate  train %.3f | valid %.3f | test %.3f | all %.3f" % tuple(mat.loc[i, "fail"].mean() for i in (idx_tr, idx_va, idx_te, mat.index)))

# random vs stratified: how far does the validation fail-rate drift from the overall rate over 500 different seeds?
dev = {"random": [], "stratified": []}
for sd in range(500):
    for name, flag in [("random", False), ("stratified", True)]:
        _, v, _ = partition(mat, strat=flag, seed=sd); dev[name].append(abs(mat.loc[v, "fail"].mean() - mat.fail.mean()))
print({k: "mean |dev| %.4f, 95th pct %.4f" % (np.mean(v), np.percentile(v, 95)) for k, v in dev.items()})

In [ ]:
# Representativeness of the three partitions (compared with the full sample)
parts = {"train": mat.loc[idx_tr], "valid": mat.loc[idx_va], "test": mat.loc[idx_te]}
numeric_cols = [c for c in mat.select_dtypes("number").columns if c not in ("fail",)]
rows = []
for n, p in parts.items():
    ks = stats.ks_2samp(p.G3, mat.G3)
    smd = ((p[numeric_cols].mean() - mat[numeric_cols].mean()) / mat[numeric_cols].std()).abs()
    rows.append((n, len(p), p.G3.mean(), p.G3.std(), ks.pvalue, smd.max(), smd.idxmax()))
display(pd.DataFrame(rows, columns=["partition","n","mean G3","sd G3","KS p (G3 vs all)","max |std. mean diff|","on variable"]).round(3))
cat_cols = [c for c in mat.columns if mat[c].dtype == object or str(mat[c].dtype) in ("str","string")]
lab = pd.Series("train", index=mat.index); lab[idx_va] = "valid"; lab[idx_te] = "test"
pv = {c: stats.chi2_contingency(pd.crosstab(mat[c], lab))[1] for c in cat_cols}
print("Chi-square p-values for partition vs categorical inputs: min %.3f (%s); share below .05: %.2f (expect ~.05 by chance)" %
      (min(pv.values()), min(pv, key=pv.get), np.mean(np.array(list(pv.values())) < .05)))

### Interpretation — Sample (written after running)
- **Target.** Mean G3 = 10.42 (sd 4.58); **32.9% fail** (G3 < 10). **38 of 395 students (9.6%) have G3 = 0**, and 25 of those had a non-zero G2 — they stopped or did not sit the final assessment. This spike will dominate the error of every model (Assess A.4).
- **Stratification matters, modestly.** Over 500 seeds, a purely random 60/20/20 split put the validation fail-rate on average 3.9 points away from the overall rate (95th percentile 10.1 points); stratified splitting made it exact. With only 79 validation rows, a 10-point swing is ~8 students — enough to flip a model ranking.
- **Representativeness.** The three partitions have near-identical G3 distributions (KS p ≈ 1). The largest standardised mean difference on any numeric input is 0.08 (train), 0.16 (valid) and 0.20 (test) standard deviations; 6% of the categorical chi-square tests have p < .05, consistent with chance. Nothing alarming, but with 80-row partitions some imbalance is unavoidable.
- **Overlap warning carried forward.** 382 students appear in both course files (author's key); this limits the cross-subject check in A.5. There is no student ID, so I cannot prove that no student is duplicated *within* math (no exact duplicate rows were found).

---
# E — EXPLORE
Rule: **explore on TRAIN only** (as StatExplore would on the training role). The validation and test partitions stay unseen, so insights, selected variables and hypotheses cannot leak.

## E.1 Univariate view, missing values, outliers

In [ ]:
tr = mat.loc[idx_tr].copy()
INPUTS = [c for c in mat.columns if c not in ("G1","G2","G3","fail")]
num_in = [c for c in INPUTS if c not in cat_cols]
print("missing values in TRAIN (all columns):", int(tr.isna().sum().sum()))
display(tr[num_in + ["G1","G2","G3"]].describe().T.round(2))
fig, ax = plt.subplots(2, 4, figsize=(17, 6.5))
for a, c in zip(ax.ravel(), ["age","absences","studytime","failures","Medu","goout","G1","G2"]):
    sns.countplot(x=tr[c], ax=a, color="#4C78A8") if tr[c].nunique() <= 12 else sns.histplot(tr[c], bins=25, ax=a, color="#4C78A8")
    a.set_title(c)
plt.tight_layout(); plt.savefig("artifacts/eda_univariate.png", dpi=110); plt.show()

In [ ]:
# outliers: Tukey fences + the G3 == 0 spike
rows = []
for c in ["absences","age","G1","G2","G3"]:
    q1, q3 = tr[c].quantile([.25, .75]); i = q3 - q1; lo, hi = q1 - 1.5 * i, q3 + 1.5 * i
    rows.append((c, int(((tr[c] < lo) | (tr[c] > hi)).sum()), lo, hi, tr[c].max()))
display(pd.DataFrame(rows, columns=["variable","n_outliers (Tukey)","lower fence","upper fence","max"]).round(1))
print("TRAIN: skewness of absences = %.2f; after log1p = %.2f" % (stats.skew(tr.absences), stats.skew(np.log1p(tr.absences))))
print("TRAIN: G3==0 count:", (tr.G3 == 0).sum(), "| G3==0 students' mean G2: %.2f vs others %.2f" % (tr.loc[tr.G3==0, "G2"].mean(), tr.loc[tr.G3>0, "G2"].mean()))

In [ ]:
# Is the G3 == 0 group linked to the absences field? (TRAIN)
ct = pd.crosstab(tr.absences == 0, tr.G3 == 0, rownames=["absences == 0"], colnames=["G3 == 0"]); display(ct)
print("share with G3==0 | absences==0: %.3f   | absences>0: %.3f" % (ct.loc[True, True] / ct.loc[True].sum(), ct.loc[False, True] / ct.loc[False].sum()))
print("mean G3 excluding G3==0 rows: absences==0 -> %.2f ; absences>0 -> %.2f" % (tr[(tr.absences==0)&(tr.G3>0)].G3.mean(), tr[(tr.absences>0)&(tr.G3>0)].G3.mean()))
print("whole math file (for the record): G3==0 rows with absences>0 =", int(((mat.G3==0)&(mat.absences>0)).sum()), "of", int((mat.G3==0).sum()))

## E.2 Bivariate view and correlations

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(18, 8))
for a, c in zip(ax.ravel(), ["failures","higher","studytime","school","Medu","goout","schoolsup","romantic"]):
    sns.boxplot(data=tr, x=c, y="G3", ax=a, color="#9ecae1"); a.set_title("G3 by " + c)
plt.tight_layout(); plt.savefig("artifacts/eda_bivariate.png", dpi=110); plt.show()

corr = tr[num_in + ["G1","G2","G3"]].corr(method="spearman")
plt.figure(figsize=(10, 8)); sns.heatmap(corr, cmap="vlag", center=0, annot=True, fmt=".1f", annot_kws={"size": 7}); plt.title("Spearman correlation (TRAIN)")
plt.tight_layout(); plt.savefig("artifacts/eda_corr.png", dpi=110); plt.show()
print("Spearman with G3:\n", corr["G3"].drop("G3").sort_values(key=abs, ascending=False).round(2).head(8).to_string())

## E.3 Variable worth (StatExplore / Variable Selection)
For each *non-grade* input on TRAIN: mutual information with `G3`, a rank test (Spearman for numeric/ordinal, Kruskal–Wallis for nominal), and a chi-square test against `fail`. Many tests on n = 237 → p-values are **Benjamini–Hochberg adjusted**. This ranks *worth*, it does not establish causality.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from statsmodels.stats.multitest import multipletests
enc = tr[INPUTS].copy()
for c in cat_cols: enc[c] = enc[c].astype("category").cat.codes
disc = np.array([(c in cat_cols) or (tr[c].nunique() <= 5) or c == "failures" for c in INPUTS])
mi = mutual_info_regression(enc, tr.G3, discrete_features=disc, random_state=SEED)
rows = []
for c, m in zip(INPUTS, mi):
    if c in cat_cols and tr[c].nunique() > 2:
        p_rank = stats.kruskal(*[g.G3.values for _, g in tr.groupby(c)]).pvalue; kind = "Kruskal"
    elif c in cat_cols:
        p_rank = stats.mannwhitneyu(*[g.G3.values for _, g in tr.groupby(c)]).pvalue; kind = "Mann-Whitney"
    else:
        p_rank = stats.spearmanr(tr[c], tr.G3).pvalue; kind = "Spearman"
    xb = tr[c] if (c in cat_cols or tr[c].nunique() <= 8) else pd.qcut(tr[c], 4, duplicates="drop")
    p_chi = stats.chi2_contingency(pd.crosstab(xb, tr.fail))[1]
    rows.append((c, m, kind, p_rank, p_chi))
worth = pd.DataFrame(rows, columns=["variable","mutual_info","rank_test","p_rank","p_chi2_fail"])
worth["q_rank (BH)"] = multipletests(worth.p_rank, method="fdr_bh")[1]
worth["q_chi2 (BH)"] = multipletests(worth.p_chi2_fail, method="fdr_bh")[1]
worth = worth.sort_values("mutual_info", ascending=False).reset_index(drop=True)
display(worth.round(4).head(15))
print("variables with BH q < .05 for G3:", worth.loc[worth["q_rank (BH)"] < .05, "variable"].tolist())
print("variables with BH q < .05 for fail:", worth.loc[worth["q_chi2 (BH)"] < .05, "variable"].tolist())
worth.to_csv("artifacts/variable_worth_train.csv", index=False)

### Interpretation — Explore (TRAIN only, n = 237)
- **No missing values; absences is heavily skewed** (skew 3.43 → 0.14 after log(1+x)); 12 absences outliers (max 75) are plausible values, not errors, so they are kept and tamed by the log transform.
- **Grades dominate everything.** Spearman ρ with G3: G2 = 0.95, G1 = 0.88; the strongest non-grade association is past `failures` (−0.38). This is the whole G1/G2 story in one number and is why the scenarios exist.
- **Variable worth (non-grade inputs).** After Benjamini–Hochberg correction, 9 variables show a significant rank association with G3 (failures, Dalc, Walc, schoolsup, higher, age, goout, Medu, Fedu), but only **failures and goout** are significant against the pass/fail flag. Effects are small — consistent with the weak early-warning models later.
- **`absences` is a trap — a data artefact, verified.** It has the *highest* mutual information (0.19) yet a Spearman p of 0.95: the relationship is non-monotone. Students with **zero** absences have the *lowest* mean G3 (8.7, see M.1). The cross-tab above explains why: **all 38 students with G3 = 0 (22 in TRAIN) have `absences == 0`** — 31% of zero-absence students in TRAIN are G3 = 0, versus 0% of students with ≥ 1 absence. Excluding the G3 = 0 rows, zero-absence students actually score *higher* (12.65 vs 11.14). So "absences = 0" is partly a recording artefact of students who did not complete the course, not 'perfect attendance'. I do not know how the school recorded it; the pattern is certain, the cause is my inference. Consequence: any model using `absences` can partly *recognise dropouts after the fact*, which will not be possible in a genuine early-year deployment — see A.4b.
- **Associations are not causes.** Several coefficients later (family support and school support both *negative*) almost certainly reflect struggling students *receiving* support (reverse causation), not support hurting grades.
- **Hypotheses carried into modelling:** (H1) past failures and age lower the grade; (H2) social/alcohol variables have small effects; (H3) nonlinear learners will beat linear ones mainly through `absences` and the G3 = 0 group (tested in A.4b); (H4) scenario A will be hard.

---
# M — MODIFY
Prepare the variables for modelling: **impute · transform · bin · derive · cluster · select**. All data-dependent steps are fit on TRAIN only and live inside sklearn pipelines so validation/test statistics never leak. The feature code is written to `src/student_features.py` so the API uses the same transformation.

## M.1 Derive and transform (Transform Variables node)
Derived: `log_absences = log(1+absences)` (skew), `alc_avg` (mean of weekday/weekend alcohol), `parent_edu` (mean of parents' education), `past_failure` (any previous class failure). The variable-clustering step (M.3) is what motivates `alc_avg` and `parent_edu`.

In [ ]:
%%writefile src/student_features.py
import numpy as np, pandas as pd

CAT = ["school","sex","address","famsize","Pstatus","Mjob","Fjob","reason","guardian","schoolsup","famsup","paid",
       "activities","nursery","higher","internet","romantic"]
NUM_RAW = ["age","Medu","Fedu","traveltime","studytime","failures","famrel","freetime","goout","Dalc","Walc","health","absences"]
NUM = [c for c in NUM_RAW if c != "absences"] + ["log_absences","alc_avg","parent_edu","past_failure"]
BASE = NUM + CAT
SCENARIOS = {"A": BASE, "B": BASE + ["G1"], "C": BASE + ["G1", "G2"]}
SCENARIO_NAMES = {"A": "A: early warning (no grades)", "B": "B: after period 1 (+G1)", "C": "C: after period 2 (+G1,G2)"}
PASS_MARK = 10

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["log_absences"] = np.log1p(d["absences"])
    d["alc_avg"] = (d["Dalc"] + d["Walc"]) / 2
    d["parent_edu"] = (d["Medu"] + d["Fedu"]) / 2
    d["past_failure"] = (d["failures"] > 0).astype(int)
    return d

def select(df: pd.DataFrame, scn: str) -> pd.DataFrame:
    return build_features(df)[SCENARIOS[scn]]

In [ ]:
sys.path.insert(0, "src")
import importlib, student_features as sf; importlib.reload(sf)
from student_features import SCENARIOS, SCENARIO_NAMES, select, build_features, CAT, NUM

XY = {}
for s in SCENARIOS:
    XY[s] = {k: select(mat.loc[i], s) for k, i in [("tr", idx_tr), ("va", idx_va), ("te", idx_te)]}
y_tr, y_va, y_te = (mat.loc[i, "G3"].values.astype(float) for i in (idx_tr, idx_va, idx_te))
f_tr, f_va, f_te = (mat.loc[i, "fail"].values for i in (idx_tr, idx_va, idx_te))
print({s: XY[s]["tr"].shape for s in XY})
print("Binning check: mean G3 by absence band (TRAIN)")
print(tr.groupby(pd.cut(tr.absences, [-1, 0, 2, 5, 10, 100]), observed=True).G3.agg(["count", "mean"]).round(2))

*Binning decision.* Mean G3 is not monotone in absences (students with zero absences include the dropout/G3=0 group), so a hand-built band would bake in a confusing shape; tree-based models find thresholds themselves, and the linear models receive `log_absences`. **No binned variable is retained.**

## M.2 Impute and encode (Impute node)
The data has no missing values, so imputation is a *no-op on this data* but is kept in the pipeline (median for numeric, mode for categorical) because production records can arrive incomplete. The cell after the pipeline definition quantifies sensitivity to missing values by injecting 10% MCAR gaps.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def make_prep(cols):
    num = [c for c in cols if c not in CAT]; cat = [c for c in cols if c in CAT]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))]), cat)])

## M.3 Variable clustering (Variable Clustering node)
Hierarchical clustering on `1 − |Spearman ρ|` among numeric non-grade inputs (average linkage), cut at |ρ| ≥ 0.5 within a cluster. Strongly related variables carry redundant information; in linear models that inflates coefficient variance.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
nm = [c for c in num_in]
rho = tr[nm].corr(method="spearman").fillna(0).values
dist = 1 - np.abs(rho); np.fill_diagonal(dist, 0)
Z = linkage(squareform(dist, checks=False), "average")
cl = fcluster(Z, t=0.5, criterion="distance")
groups = pd.Series(nm).groupby(cl).apply(list)
print("Clusters with >1 member (|rho| >= ~0.5 on average):"); [print("  ", g) for g in groups if len(g) > 1]
plt.figure(figsize=(10, 3.6)); dendrogram(Z, labels=nm, leaf_rotation=60); plt.axhline(.5, color="red", ls="--"); plt.title("Variable clustering (TRAIN)")
plt.tight_layout(); plt.savefig("artifacts/variable_clusters.png", dpi=110); plt.show()
print({c: round(float(tr[a].corr(tr[b], method="spearman")), 2) for c, a, b in [("Dalc~Walc","Dalc","Walc"), ("Medu~Fedu","Medu","Fedu"), ("goout~Walc","goout","Walc")]})

## M.4 Variable selection (Variable Selection node) — Lasso path on scenario A
A LassoCV on the standardised/encoded early-warning inputs shows which variables survive L1 shrinkage. This is **informative**, not a gate: the modelling stage keeps all inputs and lets regularisation decide (selection inside CV avoids optimistic bias).

In [ ]:
from sklearn.linear_model import LassoCV
prepA = make_prep(SCENARIOS["A"]).fit(XY["A"]["tr"])
XA = prepA.transform(XY["A"]["tr"])
las = LassoCV(cv=5, random_state=SEED, max_iter=50000).fit(XA, y_tr)
names = prepA.get_feature_names_out()
coef = pd.Series(las.coef_, index=names); kept = coef[coef != 0].sort_values(key=abs, ascending=False)
print("alpha*: %.4f | kept %d of %d encoded features" % (las.alpha_, len(kept), len(names))); print(kept.round(3).to_string())

In [ ]:
# Sensitivity of the pipeline to missing values: inject 10% MCAR into TRAIN and VALID inputs (scenario A, ridge)
from sklearn.linear_model import RidgeCV
def rmse(y, p): return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))
def inject(X, frac, rng):
    X = X.copy().astype(object)
    mask = rng.random(X.shape) < frac
    return X.mask(mask).infer_objects()
res = {}
for frac in [0.0, 0.10]:
    rng = np.random.default_rng(SEED)
    Xt, Xv = (inject(XY["A"]["tr"], frac, rng), inject(XY["A"]["va"], frac, rng)) if frac else (XY["A"]["tr"], XY["A"]["va"])
    pipe = Pipeline([("prep", make_prep(SCENARIOS["A"])), ("m", RidgeCV(alphas=np.logspace(-1, 3, 15)))]).fit(Xt, y_tr)
    res[frac] = rmse(y_va, np.clip(pipe.predict(Xv), 0, 20))
print("Valid RMSE, scenario A ridge: complete data %.3f | 10%% MCAR missing %.3f" % (res[0.0], res[0.10]))

### Interpretation — Modify
- **Variable clustering confirmed the redundancy I suspected:** `Medu~Fedu` (ρ = 0.67) and `Dalc~Walc` (ρ = 0.64) form clusters; `goout~Walc` (0.46) stays below the cut. I added `parent_edu` and `alc_avg` as combined versions but **kept the originals** — I did not test whether dropping them helps, so I make no claim that the combos improve fit. (OLS handles the exact linear dependence via minimum-norm least squares.)
- **Lasso on scenario A** kept 19 of 46 encoded columns; the largest were failures (−1.05), `famsup=yes` (−1.04), log_absences (+0.72), `schoolsup=yes` (−0.71), `famsize=LE3` (+0.70) and goout (−0.67). The positive `log_absences` sign (more absences → higher grade) is the dropout artefact from Explore (absences = 0 marks the G3 = 0 students), and the negative support signs are likely reverse causation — **do not read any of them as advice**.
- **Imputation is robust here:** injecting 10% MCAR missing values into inputs raised validation RMSE only from 4.483 to 4.533 for ridge (one seed, one model — indicative, not a robustness proof).
- **No binning kept** (see M.1) and no variable was dropped before modelling; regularisation and CV do the selection.

---
# M — MODEL
For **each information scenario** (A, B, C) fit the same set of candidate learners on TRAIN, tune hyper-parameters by 5-fold CV *inside TRAIN*, and score on VALIDATION (the Model Comparison basis). Predictions are clipped to the valid grade range 0–20.

| Learner | SAS EM analogue | Tuning (all on TRAIN, 5-fold CV) |
|---|---|---|
| Mean predictor | — (baseline) | none |
| Copy latest grade (B: G1, C: G2) | — (baseline) | none |
| OLS | Regression | none |
| Ridge | Regression (L2) | `RidgeCV`, α∈10^[-1,3] |
| Lasso | Regression / LARS | `LassoCV` |
| Decision tree | Decision Tree | depth, min leaf |
| Random forest (300 trees) | HP Forest | min leaf, max_features |
| Gradient boosting | Gradient Boosting | learning rate, depth, n_estimators, subsample |

*Neural network omitted:* with n = 237 training rows an MLP is unlikely to beat regularised linear models, and tuning it would consume the validation set's limited information.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold, RepeatedKFold, cross_val_score

cv5 = KFold(5, shuffle=True, random_state=SEED)
def reg_specs():
    return {
     "Mean (baseline)": (DummyRegressor(strategy="mean"), None),
     "OLS": (LinearRegression(), None),
     "Ridge": (RidgeCV(alphas=np.logspace(-1, 3, 15)), None),
     "Lasso": (LassoCV(cv=5, random_state=SEED, max_iter=50000), None),
     "DecisionTree": (DecisionTreeRegressor(random_state=SEED), {"m__max_depth": [2,3,4,5,6], "m__min_samples_leaf": [5,10,20]}),
     "RandomForest": (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), {"m__min_samples_leaf": [3,5,10], "m__max_features": [0.3,0.5,1.0]}),
     "GradBoost": (GradientBoostingRegressor(random_state=SEED, subsample=0.8), {"m__learning_rate": [0.03,0.1], "m__max_depth": [1,2,3], "m__n_estimators": [100,300]}),
    }

def metrics(y, p):
    y, p = np.asarray(y, float), np.clip(np.asarray(p, float), 0, 20)
    ase = float(np.mean((y - p) ** 2))
    return {"ASE": ase, "RMSE": ase ** .5, "MAE": float(np.mean(np.abs(y - p))), "R2": float(1 - ase / np.mean((y - y.mean()) ** 2))}

fitted, tuned_params = {}, {}
for s in SCENARIOS:
    for name, (est, grid) in reg_specs().items():
        pipe = Pipeline([("prep", make_prep(SCENARIOS[s])), ("m", est)])
        if grid:
            g = GridSearchCV(pipe, grid, cv=cv5, scoring="neg_root_mean_squared_error").fit(XY[s]["tr"], y_tr)
            fitted[(s, name)] = g.best_estimator_; tuned_params[(s, name)] = g.best_params_
        else:
            fitted[(s, name)] = pipe.fit(XY[s]["tr"], y_tr)
print("fitted", len(fitted), "models;  tuned params, e.g.:")
for k, v in tuned_params.items():
    if k[0] == "A": print("  ", k, v)

## Model-comparison evidence #1 — VALIDATION partition (the SAS Model Comparison basis)
Plus two baselines that need no fitting (`copy latest grade`). Validation has only 79 rows, so this table is noisy; evidence #2 (repeated CV) follows.

In [ ]:
rows = []
for s in SCENARIOS:
    for name in reg_specs():
        rows.append({"scenario": s, "model": name, **metrics(y_va, np.clip(fitted[(s, name)].predict(XY[s]["va"]), 0, 20))})
    if s in ("B", "C"):
        lastg = "G1" if s == "B" else "G2"
        rows.append({"scenario": s, "model": f"Copy {lastg} (baseline)", **metrics(y_va, XY[s]["va"][lastg].values)})
val_tbl = pd.DataFrame(rows)
display(val_tbl.pivot(index="model", columns="scenario", values="RMSE").round(3))
display(val_tbl.set_index(["scenario","model"]).round(3))

## Model-comparison evidence #2 — repeated 5×3 CV on TRAIN+VALID pooled (n = 316)
Hyper-parameters stay as tuned on TRAIN (a mild optimism for the tuned models; the ridge/lasso re-tune inside each fold). The **test partition is not touched.** Reported as mean ± SD of RMSE over 15 folds; SD is a fold-to-fold spread, so differences smaller than ≈ 1 SD should be read as ties.

In [ ]:
X_pool = {s: pd.concat([XY[s]["tr"], XY[s]["va"]]) for s in SCENARIOS}
y_pool = np.concatenate([y_tr, y_va])
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)
from sklearn.base import clone
cv_rows = []
for s in SCENARIOS:
    for name in reg_specs():
        sc = -cross_val_score(clone(fitted[(s, name)]), X_pool[s], y_pool, cv=rkf, scoring="neg_root_mean_squared_error")
        cv_rows.append({"scenario": s, "model": name, "cv_rmse_mean": sc.mean(), "cv_rmse_sd": sc.std()})
cv_tbl = pd.DataFrame(cv_rows)
cv_tbl["cv_rmse"] = cv_tbl.apply(lambda r: f"{r.cv_rmse_mean:.3f} ± {r.cv_rmse_sd:.3f}", axis=1)
display(cv_tbl.pivot(index="model", columns="scenario", values="cv_rmse"))
for s, lg in [("B","G1"),("C","G2")]:
    print(f"baseline copy-{lg} RMSE on pooled train+valid: {rmse(y_pool, X_pool[s][lg].values):.3f}")
print("baseline mean predictor on pooled data: %.3f" % rmse(y_pool, np.full_like(y_pool, y_tr.mean())))

## Pass/fail variant (classification of the at-risk class)
Same scenarios, same partition. Positive class = `fail` (G3 < 10). Metrics: ROC-AUC, PR-AUC, Brier score, and **lift@20%** (failure rate among the 20% of students ranked highest-risk ÷ overall failure rate) — the SAS Model Comparison "lift" chart read-out.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

def clf_specs():
    return {
     "Prior (baseline)": (DummyClassifier(strategy="prior"), None),
     "LogReg": (LogisticRegressionCV(Cs=10, cv=5, scoring="roc_auc", max_iter=5000), None),
     "DecisionTree": (DecisionTreeClassifier(random_state=SEED), {"m__max_depth": [2,3,4,5], "m__min_samples_leaf": [5,10,20]}),
     "RandomForest": (RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {"m__min_samples_leaf": [3,5,10], "m__max_features": [0.3,0.5,"sqrt"]}),
     "GradBoost": (GradientBoostingClassifier(random_state=SEED, subsample=0.8), {"m__learning_rate": [0.03,0.1], "m__max_depth": [1,2,3], "m__n_estimators": [100,300]}),
    }
def lift_at(y, score, frac=0.2):
    k = max(1, int(round(frac * len(y)))); top = np.argsort(-score)[:k]
    return float(np.mean(np.asarray(y)[top]) / np.mean(y))
def clf_metrics(y, p):
    return {"AUC": roc_auc_score(y, p), "PR_AUC": average_precision_score(y, p), "Brier": brier_score_loss(y, p), "lift@20%": lift_at(y, p)}

cfit = {}
for s in SCENARIOS:
    for name, (est, grid) in clf_specs().items():
        pipe = Pipeline([("prep", make_prep(SCENARIOS[s])), ("m", est)])
        cfit[(s, name)] = (GridSearchCV(pipe, grid, cv=5, scoring="roc_auc").fit(XY[s]["tr"], f_tr).best_estimator_
                           if grid else pipe.fit(XY[s]["tr"], f_tr))
rows = []
for s in SCENARIOS:
    for name in clf_specs():
        rows.append({"scenario": s, "model": name, **clf_metrics(f_va, cfit[(s, name)].predict_proba(XY[s]["va"])[:, 1])})
    if s in ("B", "C"):
        lg = "G1" if s == "B" else "G2"
        rows.append({"scenario": s, "model": f"Rule: {lg} < 10 (baseline)", **{**clf_metrics(f_va, 1 - XY[s]["va"][lg].values.astype(float) / 20), "Brier": np.nan}})
cval_tbl = pd.DataFrame(rows)
display(cval_tbl.pivot(index="model", columns="scenario", values="AUC").round(3))
display(cval_tbl.set_index(["scenario","model"]).round(3))
print("validation positives (fail):", int(f_va.sum()), "of", len(f_va), "-> AUC SE is large (~0.05)")

In [ ]:
# classification evidence #2: repeated stratified CV AUC on pooled train+valid
from sklearn.model_selection import RepeatedStratifiedKFold
f_pool = np.concatenate([f_tr, f_va])
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for s in SCENARIOS:
    for name in clf_specs():
        sc = cross_val_score(clone(cfit[(s, name)]), X_pool[s], f_pool, cv=rskf, scoring="roc_auc")
        rows.append({"scenario": s, "model": name, "cv_auc": f"{sc.mean():.3f} ± {sc.std():.3f}", "m": sc.mean()})
ccv_tbl = pd.DataFrame(rows)
display(ccv_tbl.pivot(index="model", columns="scenario", values="cv_auc"))

### Interpretation — Model (what the two evidence tables say)
**Regression RMSE (G3 sd ≈ 4.6, so the mean predictor sits at ≈ 4.6–4.9).**

| Scenario | Best learners (valid RMSE) | Pooled 5×3 CV RMSE | Strongest baseline |
|---|---|---|---|
| A (no grades) | RandomForest 4.32, DecisionTree/GradBoost 4.37 | RF 3.96 ± 0.33, GB 4.06 ± 0.30 | mean predictor: valid 4.87, pooled 4.64 ± 0.55 |
| B (+G1) | GradBoost 2.67, RF 2.74 | RF 2.42 ± 0.29, GB 2.47 ± 0.24 | copy G1: valid 3.16, pooled 2.80 |
| C (+G1,G2) | RF 1.67, GradBoost 1.68 | GB 1.77 ± 0.35, RF 1.77 ± 0.29 | copy G2: valid 2.30, pooled 2.11 |

- **Within each scenario the top tree ensembles are ties** (valid RMSE differences of 0.01–0.07; CV standard deviations 0.24–0.35). The "champion" label in A.1 is determined by a rule, not by a meaningful gap — and for scenario B the CV evidence slightly prefers RF while validation prefers GradBoost.
- **Scenario A is weak.** The best valid R² is 0.21 (RandomForest); the best pooled-CV RMSE (3.96) is ≈ 15% below the mean predictor (4.64). OLS (valid R² 0.03) is barely better than the mean; regularisation (ridge 0.15) and trees recover most of the available signal. Non-grade information carries some, but limited, information about the final grade.
- **Grades make the problem easy but not much more informative than a copy rule.** With G2, RMSE falls to ≈ 1.7, but "just copy G2" already gets 2.30 (valid) / 2.11 (pooled), and its **MAE (1.04 on valid) is essentially as good as the models' (0.99)** — the models mainly help on large errors (e.g. G3 = 0 cases).
- **Linear vs nonlinear:** trees/forests beat OLS/ridge/lasso in every scenario by 0.2–0.5 RMSE on validation. For scenario A, A.4b shows a large part of that advantage is the `absences == 0` artefact: without absence features the forest's pooled-CV RMSE rises from 3.96 to 4.39 while ridge hardly moves (4.23 → 4.31), so in A the ensemble's edge over ridge disappears (4.39 vs 4.31).

**Pass/fail classification (positive = fail; only 26 positives in validation → AUC standard error ≈ 0.05).**
- A: AUC 0.61–0.68 on validation (RF 0.68, LogReg 0.64, GB 0.64); pooled CV RF 0.687 ± 0.073, LogReg 0.671, GB 0.683 — **all within one SD of each other**.
- B: validation AUC 0.89–0.92 (GB 0.924, RF 0.920); **the one-line rule "G1 < 10" scores 0.920** — equal to the ensembles.
- C: AUC 0.96–0.97 for all learners; the rule "G2 < 10" gets 0.970 vs GradBoost 0.974.
- Lift@20% takes only a handful of distinct values on 79 rows (it is a count of ~16 students), so differences like 1.52 vs 1.71 are one student.

---
# A — ASSESS
## A.1 Champion selection (on VALIDATION only) — pre-declared rule
**Rule, fixed before looking at test:** per scenario, the champion is the non-baseline learner with the lowest **validation RMSE** (regression) or highest **validation AUC** (classification) — the analogue of Model Comparison's *Selection Statistic = Valid*. Evidence #2 (repeated CV) is reported next to it so that ties are visible; it does *not* override the rule.

In [ ]:
champ_reg, champ_clf = {}, {}
for s in SCENARIOS:
    v = val_tbl[(val_tbl.scenario == s) & ~val_tbl.model.str.contains("baseline")]
    champ_reg[s] = v.loc[v.RMSE.idxmin(), "model"]
    c = cval_tbl[(cval_tbl.scenario == s) & ~cval_tbl.model.str.contains("baseline")]
    champ_clf[s] = c.loc[c.AUC.idxmax(), "model"]
out = []
for s in SCENARIOS:
    vr = val_tbl[(val_tbl.scenario == s) & ~val_tbl.model.str.contains("baseline")].sort_values("RMSE")
    cr = cv_tbl[cv_tbl.scenario == s].set_index("model")
    top2 = vr.iloc[:2]
    out.append((s, champ_reg[s], round(top2.iloc[0].RMSE, 3), top2.iloc[1].model, round(top2.iloc[1].RMSE, 3), cr.loc[champ_reg[s], "cv_rmse"], cr.loc[top2.iloc[1].model, "cv_rmse"], champ_clf[s]))
display(pd.DataFrame(out, columns=["scenario","regression champion","valid RMSE","runner-up","runner-up valid RMSE","champion CV RMSE","runner-up CV RMSE","classification champion"]))

## A.2 Final test — each scenario's champion scored **once** on the locked TEST partition
Three pre-declared experiments (A, B, C), one test evaluation each, no re-selection afterwards. Bootstrap CIs (1,000 resamples of the 80 test rows) express test-sample noise only.

In [ ]:
def boot(y, p, fn, B=1000):
    rng = np.random.default_rng(SEED); y, p = np.asarray(y), np.asarray(p); v = []
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        try: v.append(fn(y[i], p[i]))
        except ValueError: pass
    return np.percentile(v, [2.5, 97.5])
rmse_fn = lambda a, b: rmse(a, np.clip(b, 0, 20))
TEST = {}; rows = []
for s in SCENARIOS:
    m = fitted[(s, champ_reg[s])]; p = np.clip(m.predict(XY[s]["te"]), 0, 20)
    mean_p = np.full_like(y_te, y_tr.mean())
    lo, hi = boot(y_te, p, rmse_fn)
    base = {"mean": metrics(y_te, mean_p)}
    if s != "A": base["copy"] = metrics(y_te, XY[s]["te"]["G1" if s == "B" else "G2"].values)
    mt = metrics(y_te, p)
    bp = mean_p if s == "A" else XY[s]["te"]["G1" if s == "B" else "G2"].values.astype(float)     # strongest applicable baseline
    rg = np.random.default_rng(SEED); gains = []
    for _ in range(1000):
        i = rg.integers(0, len(y_te), len(y_te)); gains.append(rmse(y_te[i], np.clip(bp[i], 0, 20)) - rmse(y_te[i], p[i]))
    g_lo, g_hi = np.percentile(gains, [2.5, 97.5])
    TEST[s] = {"model": champ_reg[s], **mt, "rmse_ci": (lo, hi), "base_mean_rmse": base["mean"]["RMSE"], "base_copy_rmse": base.get("copy", {}).get("RMSE"), "gain_ci": (g_lo, g_hi)}
    rows.append({"scenario": SCENARIO_NAMES[s], "champion": champ_reg[s], "RMSE": mt["RMSE"], "RMSE 95% CI": f"[{lo:.2f}, {hi:.2f}]", "MAE": mt["MAE"], "R2": mt["R2"],
                 "mean-baseline RMSE": base["mean"]["RMSE"], "copy-last-grade RMSE": base.get("copy", {}).get("RMSE", np.nan),
                 "RMSE gain vs best baseline, 95% CI": f"[{g_lo:+.2f}, {g_hi:+.2f}]"})
test_reg = pd.DataFrame(rows); display(test_reg.round(3))

rows = []; CTEST = {}
for s in SCENARIOS:
    m = cfit[(s, champ_clf[s])]; p = m.predict_proba(XY[s]["te"])[:, 1]
    lo, hi = boot(f_te, p, roc_auc_score)
    mt = clf_metrics(f_te, p); CTEST[s] = {"model": champ_clf[s], **mt, "auc_ci": (lo, hi)}
    rule_auc = np.nan
    if s != "A":
        lg = "G1" if s == "B" else "G2"; rs = 1 - XY[s]["te"][lg].values.astype(float) / 20
        rule_auc = roc_auc_score(f_te, rs); CTEST[s]["rule_auc"] = rule_auc
    rows.append({"scenario": SCENARIO_NAMES[s], "champion": champ_clf[s], "AUC": mt["AUC"], "AUC 95% CI": f"[{lo:.2f}, {hi:.2f}]", "PR_AUC": mt["PR_AUC"], "Brier": mt["Brier"], "lift@20%": mt["lift@20%"], "rule-baseline AUC (G<10)": rule_auc})
print("test fail rate: %.3f (n=%d, positives=%d)" % (f_te.mean(), len(f_te), f_te.sum()))
test_clf = pd.DataFrame(rows); display(test_clf.round(3))

## A.3 Lift / gains (pass-fail) and calibration
Calibration and gains for the *deployable* scenario A classifier are shown on **out-of-fold predictions over train+valid** (n = 316, ≈ 100 positives) because the 80-row test set is too small for a stable calibration curve; the test-set gains curve is shown alongside.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.calibration import calibration_curve
sA = "A"; mA = cfit[(sA, champ_clf[sA])]
oofA = cross_val_predict(clone(mA), X_pool[sA], f_pool, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
pt = mA.predict_proba(XY[sA]["te"])[:, 1]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
fr, mp = calibration_curve(f_pool, oofA, n_bins=6, strategy="quantile"); ax[0].plot(mp, fr, "o-"); ax[0].plot([0,1],[0,1],"k--")
ax[0].set_title(f"Calibration, scenario A ({champ_clf[sA]}), OOF n={len(f_pool)}"); ax[0].set_xlabel("predicted P(fail)"); ax[0].set_ylabel("observed")
for lbl, y_, p_ in [("OOF train+valid", f_pool, oofA), ("TEST", f_te, pt)]:
    o = np.argsort(-p_); g = np.cumsum(y_[o]) / y_.sum(); ax[1].plot(np.arange(1, len(o)+1) / len(o), g, label=lbl)
ax[1].plot([0,1],[0,1],"k--", label="random"); ax[1].set_title("Cumulative gains, scenario A"); ax[1].legend()
for s in SCENARIOS:
    o = np.argsort(-cfit[(s, champ_clf[s])].predict_proba(XY[s]["te"])[:, 1]); g = np.cumsum(f_te[o]) / f_te.sum()
    ax[2].plot(np.arange(1, len(o)+1) / len(o), g, label=f"{s}: {champ_clf[s]}")
ax[2].plot([0,1],[0,1],"k--"); ax[2].set_title("TEST gains by scenario"); ax[2].legend()
plt.tight_layout(); plt.savefig("artifacts/gains_calibration.png", dpi=120); plt.show()
print("OOF scenario A: AUC %.3f | Brier %.3f (prior-only Brier %.3f) | lift@20%% %.2f" % (roc_auc_score(f_pool, oofA), brier_score_loss(f_pool, oofA), brier_score_loss(f_pool, np.full(len(f_pool), f_pool.mean())), lift_at(f_pool, oofA)))
for q in [0.1, 0.2, 0.3]:
    k = int(q * len(f_pool)); top = np.argsort(-oofA)[:k]
    print(f"  flag top {int(q*100)}%: precision {f_pool[top].mean():.2f}, recall {f_pool[top].sum()/f_pool.sum():.2f}")

## A.4 Residual diagnostics (regression, out-of-fold on TRAIN+VALID)

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(17, 8.5))
for r, s in enumerate(["A", "C"]):
    oof = np.clip(cross_val_predict(clone(fitted[(s, champ_reg[s])]), X_pool[s], y_pool, cv=cv5), 0, 20); res_ = y_pool - oof
    ax[r,0].scatter(oof, res_, s=12, c=(y_pool == 0), cmap="coolwarm"); ax[r,0].axhline(0, color="k"); ax[r,0].set_title(f"{s} ({champ_reg[s]}): residual vs fitted (red = G3=0)")
    stats.probplot(res_, plot=ax[r,1]); ax[r,1].set_title("normal Q-Q of residuals")
    ax[r,2].scatter(y_pool, oof, s=12); ax[r,2].plot([0,20],[0,20],"k--"); ax[r,2].set_title("observed vs predicted"); ax[r,2].set_xlabel("G3")
    nz = y_pool > 0
    print(f"{s}: OOF RMSE all {rmse(y_pool, oof):.3f} | excluding G3==0 rows {rmse(y_pool[nz], oof[nz]):.3f} | G3==0 rows only {rmse(y_pool[~nz], oof[~nz]):.3f} (n={(~nz).sum()}) | Shapiro p={stats.shapiro(res_).pvalue:.1e}")
plt.tight_layout(); plt.savefig("artifacts/residual_diagnostics.png", dpi=110); plt.show()

## A.4b Robustness — does scenario A depend on the `absences == 0` artefact?
Explore showed that every G3 = 0 student has `absences == 0`. In a real early-year deployment that pattern would not yet exist (a student who will drop out has not yet shown a 'zero' for a later period). So I re-estimate scenario A **without** `log_absences` (pooled 5×3 CV, same tuned hyper-parameters; the preprocessing is re-fitted inside each fold).

In [ ]:
colsNA = [c for c in SCENARIOS["A"] if c != "log_absences"]
rowsr = []
for lbl, cols in [("A (with absences)", SCENARIOS["A"]), ("A minus absences", colsNA)]:
    for nm in ["RandomForest", "GradBoost", "Ridge"]:
        est = clone(fitted[("A", nm)].named_steps["m"]); pipe = Pipeline([("prep", make_prep(cols)), ("m", est)])
        r = -cross_val_score(pipe, X_pool["A"][cols], y_pool, cv=rkf, scoring="neg_root_mean_squared_error")
        rowsr.append((lbl, nm, "RMSE", r.mean(), r.std()))
    for nm in ["RandomForest", "LogReg"]:
        est = clone(cfit[("A", nm)].named_steps["m"]); pipe = Pipeline([("prep", make_prep(cols)), ("m", est)])
        r = cross_val_score(pipe, X_pool["A"][cols], f_pool, cv=rskf, scoring="roc_auc")
        rowsr.append((lbl, nm, "fail AUC", r.mean(), r.std()))
rb = pd.DataFrame(rowsr, columns=["scenario A variant", "model", "metric", "mean", "sd"]).round(3); display(rb)
print("mean-predictor pooled RMSE (reference): %.3f" % rmse(y_pool, np.full_like(y_pool, y_tr.mean())))

## A.5 Cross-subject sanity check — train on math, score Portuguese (scenario A)
**Caveat:** about 382 students appear in both files (see S.1), so Portuguese rows are *not independent* of the math training rows — this is a stability check of the feature→grade relationship, **not** a clean external validation.

In [ ]:
por["fail"] = (por["G3"] < PASS_MARK).astype(int)
Xp = select(por, "A"); yp = por["G3"].values.astype(float)
mA_reg = fitted[("A", champ_reg["A"])]
rows = [("trained on math TRAIN, scored on ALL Portuguese", rmse(yp, np.clip(mA_reg.predict(Xp), 0, 20)))]
rows.append(("math TRAIN mean predictor on Portuguese", rmse(yp, np.full_like(yp, y_tr.mean()))))
rows.append(("Portuguese own mean predictor (in-sample mean)", rmse(yp, np.full_like(yp, yp.mean()))))
sc = -cross_val_score(clone(mA_reg), Xp, yp, cv=RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED), scoring="neg_root_mean_squared_error")
rows.append((f"{champ_reg['A']} trained & CV'd inside Portuguese (5x3)", sc.mean()))
xs = pd.DataFrame(rows, columns=["setting","RMSE"]); display(xs.round(3))
print("mean G3: math %.2f vs Portuguese %.2f ; fail rate: math %.3f vs Portuguese %.3f" % (mat.G3.mean(), por.G3.mean(), mat.fail.mean(), por.fail.mean()))
pp = cfit[("A", champ_clf["A"])].predict_proba(Xp)[:, 1]
print("fail AUC trained on math, scored on Portuguese: %.3f" % roc_auc_score(por.fail, pp))

### Interpretation — Assess (test set, used once per scenario)

| Scenario | Champion (valid rule) | Test RMSE [95% CI] | Strongest baseline RMSE | RMSE gain over it [95% CI] | Fail AUC [95% CI] | Rule-baseline AUC |
|---|---|---|---|---|---|---|
| A | RandomForest | 4.18 [3.16, 5.06] (R² = 0.06) | mean 4.31 | **[−0.57, +0.91]** | 0.66 [0.54, 0.79] | — |
| B | GradBoost | 2.64 [2.05, 3.30] | copy G1: 2.80 | **[−0.34, +0.63]** | 0.94 [0.89, 0.98] | 0.93 |
| C | RandomForest (clf: GradBoost) | 1.24 [0.96, 1.54] | copy G2: 1.53 | **[−0.20, +0.74]** | 0.99 [0.96, 1.00] | 0.98 |

**Headline conclusions, stated carefully**
1. **On the test set none of the three champions is statistically distinguishable from its strongest baseline** — every paired-bootstrap interval for the RMSE gain contains zero. The point estimates favour the models (and the pooled-CV evidence on 316 students agrees in direction: ≈ 15% RMSE reduction over the mean in A; 2.4–2.5 vs 2.80 in B; 1.77 vs 2.11 in C), but with 79 test rows I cannot claim the models beat "copy the last grade" for B and C. A larger sample or a different school year would be needed.
2. **Scenario A (the meaningful early-warning case) is modest.** Test AUC 0.66 (CI includes 0.54) and test R² ≈ 0.06 are far below the CV R² ≈ 0.27 implied by pooled CV (1 − (3.96/4.64)²). Two honest reasons: the CV hyper-parameters were tuned on TRAIN rows that appear in the pooled folds (optimistic), and the test partition is small. On the 316-student out-of-fold predictions (AUC 0.686, Brier 0.198 vs 0.221 for always predicting the base rate), flagging the **top 20%** gives precision 0.59 (base rate 0.33) and recall 0.36; top 30% gives precision 0.55, recall 0.50. That is a *prioritisation* aid at best, not a reliable predictor.
3. **The G1/G2 answer.** Adding G1 cuts RMSE from ≈ 4.2 to ≈ 2.6 and AUC from 0.66 to 0.94; adding G2 reaches RMSE 1.24 and AUC 0.99. This is "good" mostly because earlier grades *are* earlier measurements of the same quantity: a rule "G2 < 10" already achieves AUC 0.98. Scenario C should be read as an upper bound / sanity check, scenario A as the real early-warning problem, and scenario B as the practical compromise (a mid-year intervention still has time to act).
4. **Calibration** (scenario A, out-of-fold, 6 quantile bins): predicted probabilities only span ≈ 0.18–0.56 — the model never expresses high confidence — and the curve is near the diagonal but non-monotone (a bin predicted at ≈ 0.33 shows ≈ 0.22 observed). The Brier gain over the prior is small (0.198 vs 0.221). The out-of-fold gains curve is above the random diagonal; the TEST gains curve for A follows it more loosely.
4b. **Robustness (A.4b).** Removing the absence feature, pooled-CV RMSE is RF 4.39, GradBoost 4.47, Ridge 4.31 against the mean predictor 4.64 (≈ 7% lower for ridge), and fail AUC stays ≈ 0.67 (RF 0.665, LogReg 0.672 vs 0.687 / 0.671 with absences). So the *ranking* signal for pass/fail survives; the regression advantage of trees mostly does not. **The realistic early-warning performance is therefore the A-minus-absences figures, not the headline test numbers.**
5. **Residuals.** Shapiro–Wilk rejects normality of residuals for both A and C (p = 1e-4 and 5e-19), so OLS-style prediction intervals would be unreliable; the residuals are driven by **G3 = 0 students**: out-of-fold RMSE for scenario A is 4.04 overall but 3.40 excluding G3 = 0 rows and **7.67** on those 32 rows; for scenario C the figures are 1.87 / 1.36 / 4.26. No model predicts a student abandoning the course (even with G2 known), apart from the absence artefact in A.
6. **Cross-subject check (not independent: ~382 overlapping students).** The math-trained scenario-A forest scores RMSE 3.22 on Portuguese vs 3.56 for the math-train mean and 3.23 for Portuguese's own mean, i.e. it does **not** beat a Portuguese-specific constant; a forest trained inside Portuguese gets 2.67 (CV). Fail AUC when transferring is 0.775 (Portuguese fail rate is only 15% vs 33% in math). So the *ranking* signal partly transfers; the *level* of grades does not.

**Limitations.** n = 395 (80-row partitions); one region/school year; hyper-parameter grids are small; test set was evaluated once per scenario (three pre-declared experiments), but I did look at it for the table above only after champions were fixed; no fairness audit although sex, age and family background are inputs; observational data, so no causal reading of any coefficient.

---
# Deployment artifacts (SEMMA's Score node → production)
SEMMA stops at Assess; the SAS Score node is how its results are used. Here: (1) one fitted pipeline per scenario, (2) a FastAPI scorer that chooses the scenario from which grades are supplied, (3) Docker, (4) SAS-style score code for the linear model, (5) a model card whose numbers are written **by this notebook** from the results above.

In [ ]:
import joblib
art = {"regressors": {s: fitted[(s, champ_reg[s])] for s in SCENARIOS}, "classifiers": {s: cfit[(s, champ_clf[s])] for s in SCENARIOS},
       "reg_names": champ_reg, "clf_names": champ_clf, "pass_mark": PASS_MARK, "trained_at": datetime.datetime.now().isoformat(timespec="seconds"),
       "train_rows": len(idx_tr), "seed": SEED, "test_reg": {s: {k: v for k, v in TEST[s].items()} for s in TEST}, "test_clf": CTEST}
joblib.dump(art, "artifacts/student_pipelines.joblib")
print({k: art[k] for k in ["reg_names","clf_names","trained_at","train_rows"]})

In [ ]:
%%writefile deploy/app.py
import sys, os, joblib, pandas as pd, numpy as np
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from typing import Optional
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from student_features import select, SCENARIO_NAMES

ART = joblib.load(os.environ.get("MODEL_PATH", os.path.join(os.path.dirname(__file__), "..", "artifacts", "student_pipelines.joblib")))
app = FastAPI(title="Student grade early-warning", version="1.0")

class Student(BaseModel):
    school: str; sex: str; age: int; address: str; famsize: str; Pstatus: str; Medu: int; Fedu: int; Mjob: str; Fjob: str
    reason: str; guardian: str; traveltime: int; studytime: int; failures: int; schoolsup: str; famsup: str; paid: str
    activities: str; nursery: str; higher: str; internet: str; romantic: str; famrel: int; freetime: int; goout: int
    Dalc: int; Walc: int; health: int; absences: int
    G1: Optional[int] = None   # supply after period 1 to use scenario B
    G2: Optional[int] = None   # supply G1 and G2 after period 2 to use scenario C

@app.get("/health")
def health():
    return {"status": "ok", "trained_at": ART["trained_at"], "regressors": ART["reg_names"], "classifiers": ART["clf_names"]}

def _scenario(s: Student) -> str:
    if s.G2 is not None and s.G1 is None: raise HTTPException(422, "G2 given without G1")
    return "C" if s.G2 is not None else ("B" if s.G1 is not None else "A")

@app.post("/predict")
def predict(s: Student):
    scn = _scenario(s)
    df = pd.DataFrame([s.model_dump()])
    X = select(df, scn)
    g3 = float(np.clip(ART["regressors"][scn].predict(X)[0], 0, 20))
    pf = float(ART["classifiers"][scn].predict_proba(X)[0, 1])
    return {"scenario": SCENARIO_NAMES[scn], "predicted_G3": round(g3, 2), "p_fail": round(pf, 3), "model": ART["reg_names"][scn],
            "caveat": "Statistical estimate from n=316 students; use to prioritise support, not to decide outcomes."}

In [ ]:
sys.path.insert(0, "deploy")
import app as app_mod; importlib.reload(app_mod)
rec = mat.loc[idx_te[0]].drop(["G3", "fail"]).to_dict()
for k in rec:
    if isinstance(rec[k], (np.integer,)): rec[k] = int(rec[k])
print("true G3:", mat.loc[idx_te[0], "G3"], "| health:", app_mod.health()["regressors"])
r_A = app_mod.predict(app_mod.Student(**{k: v for k, v in rec.items() if k not in ("G1", "G2")}));  print(r_A)
r_B = app_mod.predict(app_mod.Student(**{k: v for k, v in rec.items() if k != "G2"}));            print(r_B)
r_C = app_mod.predict(app_mod.Student(**rec));                                                    print(r_C)
try: app_mod.predict(app_mod.Student(**{k: v for k, v in rec.items() if k != "G1"}))
except Exception as e: print("G2 without G1 ->", type(e).__name__, getattr(e, "status_code", ""), getattr(e, "detail", e))
# API must reproduce the notebook's predictions exactly
for s, r in zip("ABC", [r_A, r_B, r_C]):
    assert abs(r["predicted_G3"] - float(np.clip(fitted[(s, champ_reg[s])].predict(XY[s]["te"].iloc[[0]])[0], 0, 20))) < 0.006
print("API == notebook predictions: OK")

In [ ]:
%%writefile deploy/Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY deploy/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src ./src
COPY artifacts/student_pipelines.joblib ./artifacts/student_pipelines.joblib
COPY deploy/app.py ./deploy/app.py
EXPOSE 8000
CMD ["uvicorn", "deploy.app:app", "--host", "0.0.0.0", "--port", "8000"]

In [ ]:
import sklearn, fastapi
open("deploy/requirements.txt", "w").write("\n".join([f"scikit-learn=={sklearn.__version__}", f"pandas=={pd.__version__}", f"numpy=={np.__version__}",
    "joblib", f"fastapi=={fastapi.__version__}", "uvicorn[standard]", "pydantic>=2"]) + "\n")
print(open("deploy/requirements.txt").read())

## Score-code export (SAS Score node style)
SAS's Score node emits a DATA step with the fitted equation. Tree ensembles do not translate compactly, so I export the **ridge regression** for each scenario as a DATA-step — a transparent, auditable model that can be compared with SAS's own Regression-node output. Verified here by recomputing predictions from the printed coefficients and comparing to the sklearn pipeline (max absolute difference reported). **The SAS syntax itself was not executed in SAS.**

In [ ]:
def ridge_scorecode(s):
    pipe = Pipeline([("prep", make_prep(SCENARIOS[s])), ("m", RidgeCV(alphas=np.logspace(-1, 3, 15)))]).fit(XY[s]["tr"], y_tr)
    prep, m = pipe.named_steps["prep"], pipe.named_steps["m"]
    num_cols = prep.transformers_[0][2]; cat_cols_ = prep.transformers_[1][2]
    imp, sc = prep.named_transformers_["num"].named_steps["imp"], prep.named_transformers_["num"].named_steps["sc"]
    oh = prep.named_transformers_["cat"].named_steps["oh"]
    terms, k = [], 0                                   # terms: (kind, column, level, mean, scale, impute, coef)
    for j, c in enumerate(num_cols):
        terms.append(("num", c, None, sc.mean_[j], sc.scale_[j], imp.statistics_[j], m.coef_[k])); k += 1
    for j, c in enumerate(cat_cols_):
        drop = None if oh.drop_idx_ is None else oh.drop_idx_[j]
        for li, lev in enumerate(oh.categories_[j]):
            if drop is not None and li == drop: continue
            terms.append(("cat", c, lev, None, None, None, m.coef_[k])); k += 1
    assert k == len(m.coef_)
    return pipe, m.intercept_, terms

def eval_terms(df, b0, terms):
    out = np.full(len(df), b0, dtype=float)
    for kind, c, lev, mu, sd, im, w in terms:
        if kind == "num": out += w * ((df[c].astype(float).fillna(im) - mu) / sd).values
        else: out += w * (df[c] == lev).astype(float).values
    return out

def to_sas(s, b0, terms):
    L = [f"/* Score code (ridge regression), scenario {SCENARIO_NAMES[s]} — generated {datetime.date.today()} from semma_student_performance.ipynb",
         "   NOT executed in SAS; arithmetic verified against scikit-learn in Python. Input: raw columns; derived features computed first. */",
         "log_absences = log(1 + absences); alc_avg = (Dalc + Walc)/2; parent_edu = (Medu + Fedu)/2; past_failure = (failures > 0);",
         f"P_G3 = {b0:.6f}"]
    for kind, c, lev, mu, sd, im, w in terms:
        L.append(f"   + {w:.6f} * ((coalesce({c}, {im:.6f}) - {mu:.6f}) / {sd:.6f})" if kind == "num" else f"   + {w:.6f} * ({c} = '{lev}')")
    L.append(";\nP_G3 = min(20, max(0, P_G3));")
    return "\n".join(L)

for s in SCENARIOS:
    pipe, b0, terms = ridge_scorecode(s)
    diff = np.abs(eval_terms(XY[s]["te"], b0, terms) - pipe.predict(XY[s]["te"])).max()
    open(f"artifacts/score_code_{s}.sas.txt", "w").write(to_sas(s, b0, terms))
    print(f"scenario {s}: {len(terms)} terms | max |score-code - sklearn| on test = {diff:.2e}")
print(open("artifacts/score_code_A.sas.txt").read()[:700], "...")

In [ ]:
card = f'''# Model card — Student grade early-warning (SEMMA project)

**Intended use.** Prioritise *support outreach* (tutoring, check-ins) for students likely to fail, using information available at three points in the year (scenarios A/B/C). **Not** for grading, tracking, admissions or any decision about a student without human review.

**Data.** UCI Student Performance, math course, n = 395 (two Portuguese schools, 2005–06). Partition 60/20/20 stratified on fail = (G3 < 10): train {len(idx_tr)} / valid {len(idx_va)} / test {len(idx_te)}. Pipelines are fit on TRAIN only.

**Models (champions chosen on validation).**
{chr(10).join(f"- {SCENARIO_NAMES[s]}: regression = {champ_reg[s]}; fail classifier = {champ_clf[s]}" for s in SCENARIOS)}

**Test performance (one evaluation per scenario, n = {len(idx_te)}).**

| Scenario | RMSE (95% CI) | mean-baseline RMSE | copy-last-grade RMSE | fail AUC (95% CI) | lift@20% |
|---|---|---|---|---|---|
''' + "\n".join(f"| {s} | {TEST[s]['RMSE']:.2f} [{TEST[s]['rmse_ci'][0]:.2f}, {TEST[s]['rmse_ci'][1]:.2f}] | {TEST[s]['base_mean_rmse']:.2f} | {('%.2f' % TEST[s]['base_copy_rmse']) if TEST[s]['base_copy_rmse'] is not None else '—'} | {CTEST[s]['AUC']:.2f} [{CTEST[s]['auc_ci'][0]:.2f}, {CTEST[s]['auc_ci'][1]:.2f}] | {CTEST[s]['lift@20%']:.2f} |" for s in SCENARIOS) + '''

**Limitations.** Small sample (test n = 79; intervals are wide); two schools in one region, one school year (as reported by the dataset authors), so generalisation elsewhere is untested; G3 = 0 (dropout/no-show) rows are poorly predicted by any model, and in this file *every* such student has absences = 0 — a recording artefact the tree models can exploit, which will not exist in a real early-year deployment (without absence features the early-warning model's RMSE is only ≈ 7% below a constant, AUC ≈ 0.67); sensitive attributes (sex, age, address, parental education/job, romantic status) are model inputs and could encode socioeconomic bias — *no fairness audit was performed*; Portuguese-course rows overlap ~382 students with math, so the cross-subject check is not independent. Pipelines were fit on TRAIN only (not refit on train+valid) to keep the test estimate honest; refit before production use.

**Maintenance.** Re-fit each school year; monitor score distribution and, once final grades are known, RMSE/AUC by scenario.

**Process notes.** Python re-implementation of the SAS SEMMA workflow; SAS was not run. Generated by `semma_student_performance.ipynb`.
'''
open("artifacts/model_card.md", "w").write(card); print(card)

---
# Summary against SEMMA's five stages
Stage-by-stage conclusions, the answer to the G1/G2 question and the limitations are given in the interpretation blocks above (written after execution). Reproduce everything with *Runtime → Run all*; seeds are fixed (42).

### Final reading across the five SEMMA stages
- **Sample** — stratified 60/20/20 on pass/fail removes up to ±10 points of fail-rate drift that random splitting can produce at this n; partitions were representative.
- **Explore** — past failures are the main non-grade signal; **every G3 = 0 student has absences = 0** (a recording artefact that inflates tree models' apparent early-warning skill); much of the "support" signal is reverse causation.
- **Modify** — log-transform for absences, two combined variables, no binning; imputation is a no-op on this data.
- **Model** — tree ensembles beat linear models; top ensembles tie each other.
- **Assess** — with grades the task is nearly trivial and a copy-last-grade rule is a strong baseline; **without grades, the model is only a modest ranking aid** (test AUC 0.66; pooled-CV AUC ≈ 0.67–0.69, ≈ 0.67 without absences; RMSE ≈ 7% below the mean predictor without absences). All test-set improvements over baselines are within noise.
- **Deployment** — the scorer chooses scenario A/B/C from the grades supplied and reproduces notebook predictions exactly; fitted on TRAIN only.

**What I would do next:** more data (additional years/schools), a prospective test of the early-warning list with human follow-up, a fairness audit across sex/address/parent education, and a SAS Enterprise Miner re-run to compare against this Python re-implementation.